# 04 Eval And Generation Walkthrough

This notebook explains what happens **after training saves a checkpoint**.

We will connect toy examples to the real nanochat flow:

```text
checkpoint files -> load_model(...) -> Engine.generate(...) -> samples
checkpoint files -> load_model(...) -> evaluate_bpb(...) -> train/val BPB
checkpoint files -> load_model(...) -> evaluate_core(...) -> CORE benchmark
```

This is the notebook to understand before spending real GPU money, because it teaches how to inspect whether a run is actually improving.

In [ ]:
from pathlib import Path
import math
import os
import sys

repo_root = Path.cwd()
if not ((repo_root / "nanochat").exists() and (repo_root / "pyproject.toml").exists()):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "nanochat").exists() and (candidate / "pyproject.toml").exists():
            repo_root = candidate
            break

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

missing = []
for package in ["torch", "rustbpe", "tiktoken", "tokenizers"]:
    try:
        __import__(package)
    except Exception:
        missing.append(package)

if missing:
    raise RuntimeError(
        "Missing dependencies: " + ", ".join(missing) + "\n"
        "From the repo root run:\n"
        "  uv sync --extra cpu --group dev\n"
        "or, on CUDA machines:\n"
        "  uv sync --extra gpu --group dev\n"
        "Then reopen this notebook with the nanochat (.venv) kernel."
    )

import torch

from nanochat.common import get_base_dir, COMPUTE_DTYPE, COMPUTE_DTYPE_REASON

print(f"repo root: {repo_root}")
print(f"base dir:  {get_base_dir()}")
print(f"torch:     {torch.__version__}")
print(f"cuda:      {torch.cuda.is_available()}")
print(f"dtype:     {COMPUTE_DTYPE} ({COMPUTE_DTYPE_REASON})")

## Step 1. What `base_eval.py` does

`scripts/base_eval.py` has three separate jobs:

| Eval mode | What it answers | Real code path |
|---|---|---|
| `sample` | What does the model say if we ask it to continue text? | `load_model(...)` -> `Engine.generate_batch(...)` |
| `bpb` | How surprised is the model by train/val text? | dataloader -> `model(x, y)` -> `evaluate_bpb(...)` |
| `core` | Can the base model solve benchmark prompts? | CORE task renderer -> token losses -> accuracy |

Important: `sample` is qualitative. `bpb` and `core` are quantitative.

In [ ]:
print("base_eval.py mental model:\n")
print("1. load checkpoint + tokenizer")
print("2. if --eval=sample: generate text with Engine")
print("3. if --eval=bpb: compute loss per byte on train/val text")
print("4. if --eval=core: run benchmark tasks and score accuracy")
print("5. write results into the nanochat report/base cache")

## Step 2. Find local checkpoints

Training saved files like:

```text
~/.cache/nanochat/base_checkpoints/d4/model_002500.pt
~/.cache/nanochat/base_checkpoints/d4/meta_002500.json
~/.cache/nanochat/base_checkpoints/d4/optim_002500_rank0.pt
```

`base_eval.py` only needs the model checkpoint and metadata. The optimizer checkpoint is for **resuming training**, not for evaluation.

In [ ]:
base_dir = Path(get_base_dir())
checkpoint_root = base_dir / "base_checkpoints"

selected_model_tag = None
selected_step = None

print(f"checkpoint root: {checkpoint_root}")

if not checkpoint_root.exists():
    print("No base checkpoints found yet.")
    print("Train first, for example with scripts.base_train, then rerun this cell.")
else:
    rows = []
    for tag_dir in sorted(p for p in checkpoint_root.iterdir() if p.is_dir()):
        steps = []
        for path in tag_dir.glob("model_*.pt"):
            try:
                steps.append(int(path.stem.split("_")[-1]))
            except ValueError:
                pass
        if steps:
            rows.append((tag_dir.name, max(steps), len(steps)))

    if not rows:
        print("Checkpoint directory exists, but no model_*.pt files were found.")
    else:
        print("Available checkpoint tags:")
        for tag, last_step, count in rows:
            print(f"  {tag:12s} latest step: {last_step:6d}   saved models: {count}")

        selected_model_tag, selected_step, _ = max(rows, key=lambda row: row[1])
        print("\nSelected for notebook demo:")
        print(f"  --model-tag={selected_model_tag}")
        print(f"  --step={selected_step}")

## Step 3. Load a checkpoint like `base_eval.py`

Real `base_eval.py` does this for nanochat checkpoints:

```python
model, tokenizer, meta = load_model("base", device, phase="eval", model_tag=..., step=...)
```

That gives us:

- `model`: the GPT weights loaded from `model_XXXXXX.pt`
- `tokenizer`: loaded through `get_tokenizer()` from `tokenizer.pkl`
- `meta`: config and training metadata loaded from `meta_XXXXXX.json`

In [ ]:
from nanochat.checkpoint_manager import load_model

model = None
tokenizer = None
meta = None

if selected_model_tag is None:
    print("Skipping checkpoint load because no checkpoint was found.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading on device: {device}")
    model, tokenizer, meta = load_model(
        "base",
        device,
        phase="eval",
        model_tag=selected_model_tag,
        step=selected_step,
    )
    print("Loaded checkpoint.")
    print("model config:", meta["model_config"])
    print("checkpoint step:", meta["step"])
    print("tokenizer vocab size:", tokenizer.get_vocab_size())

## Step 4. Generation starts with logits

A GPT model does not directly output text. It outputs **logits**: one score per possible next token.

Then sampling code chooses the next token.

In nanochat this happens in `nanochat/engine.py`:

```text
tokens -> tensor ids -> model.forward(...) -> logits -> sample_next_token(...) -> next token id
```

Temperature controls how deterministic or random this choice is:

- `temperature=0`: greedy, always pick the largest logit
- `temperature=1`: sample from probabilities
- lower temperature: safer / more repetitive
- higher temperature: more random / more chaotic

In [ ]:
toy_vocab = ["Paris", "the", "protein", "blue", ".", "Friday"]
toy_logits = torch.tensor([2.0, 4.5, 1.7, 0.8, 2.3, 1.0])

def show_next_token_distribution(logits, temperature):
    if temperature == 0:
        chosen = torch.argmax(logits).item()
        print(f"temperature=0 -> greedy token: {toy_vocab[chosen]!r}")
        return
    probs = torch.softmax(logits / temperature, dim=-1)
    print(f"temperature={temperature}")
    for token, prob in sorted(zip(toy_vocab, probs.tolist()), key=lambda x: -x[1]):
        print(f"  {token:8s} {prob:.3f}")

show_next_token_distribution(toy_logits, temperature=0)
print()
show_next_token_distribution(toy_logits, temperature=1.0)
print()
show_next_token_distribution(toy_logits, temperature=2.0)

## Step 5. Generate from your real checkpoint

This mirrors the `sample` section of `scripts/base_eval.py`.

Real relation to nanochat:

```text
prompt string -> tokenizer(prompt, prepend="<|bos|>") -> token ids
token ids -> Engine.generate_batch(...) -> generated token ids
generated token ids -> tokenizer.decode(...) -> text
```

The output may still be wrong or silly. That is not a bug. Small base models learn text patterns before they learn reliable facts.

In [ ]:
from nanochat.engine import Engine

if model is None:
    print("Skipping real generation because no checkpoint is loaded.")
else:
    engine = Engine(model, tokenizer)
    prompts = [
        "The capital of France is",
        "The opposite of hot is",
        "If 5*x + 3 = 13, then x is",
    ]
    for prompt in prompts:
        tokens = tokenizer(prompt, prepend="<|bos|>")
        generated, masks = engine.generate_batch(
            tokens,
            num_samples=1,
            max_tokens=16,
            temperature=0.0,
        )
        print("-" * 80)
        print(tokenizer.decode(generated[0]))

## Step 6. Why generation uses a KV cache

Without a KV cache, every new generated token would recompute attention for the whole prompt again.

With a KV cache:

```text
prefill: prompt tokens -> cache keys/values for all prompt positions
decode:  one new token -> append one new key/value -> predict next token
```

This is why generation has extra machinery that training does not have.

Your 2080 Ti bug was here: the model used `float32`, but the old generation cache assumed every CUDA GPU should use `bfloat16`. The fix is that `Engine` now uses `COMPUTE_DTYPE` for the cache, matching the model.

In [ ]:
B = 1
prompt_len = 7
num_layers = 4
n_kv_head = 2
head_dim = 128

print("Toy KV cache shape used by nanochat Engine:")
print(f"  k_cache: ({num_layers}, {B}, max_seq_len, {n_kv_head}, {head_dim})")
print(f"  v_cache: ({num_layers}, {B}, max_seq_len, {n_kv_head}, {head_dim})")
print()
print("During prefill:")
print(f"  cache positions 0..{prompt_len - 1} are filled from the prompt")
print()
print("During decode:")
print(f"  the next token appends position {prompt_len}")
print()
print("Current notebook compute dtype:")
print(f"  {COMPUTE_DTYPE} ({COMPUTE_DTYPE_REASON})")
print("Engine KV cache should use this same dtype.")

## Step 7. What BPB measures

`BPB` means **bits per byte**.

Short version: lower `BPB` means the model is less surprised by real text, so it is doing better.

Normal training loss is measured per token. But different tokenizers can make longer or shorter tokens, so per-token loss can be misleading.

BPB asks:

```text
How many bits of surprise does the model have per byte of real text?
```

Real nanochat relation:

```text
dataloader -> x, y
model(x, y, loss_reduction='none') -> loss for each target token
token_bytes[y] -> how many bytes each target token represents
total loss / total bytes -> BPB
```

Special tokens like `<|bos|>` count as `0` bytes, because they are training/control markers, not real text bytes.

Where nanochat stores this information:

- `scripts/base_train.py` prints `Step XXXXX | Validation bpb: ...` during training.
- checkpoint metadata saves the latest value as `val_bpb` in `meta_XXXXXX.json`.
- the training report saves `Minimum validation bpb` and `Final validation bpb`.
- W&B logs it as `val/bpb` if W&B logging is enabled.

So yes: BPB is important training information. It is not part of the model weights, but it tells you how good that checkpoint was.

In [ ]:
# Toy version of nanochat/loss_eval.py:evaluate_bpb

# Imagine these are target token ids from y.
# 0 is a special token, so it contributes 0 real text bytes.
target_ids = torch.tensor([1, 2, 0, 3])

# Per-target loss in nats. Real nanochat gets this from model(x, y, loss_reduction='none').
loss_nats = torch.tensor([0.7, 1.1, 0.4, 2.0])

# token_bytes[id] says how many UTF-8 bytes this token represents.
token_bytes = torch.tensor([0, 3, 1, 6])

bytes_per_target = token_bytes[target_ids]
counted = bytes_per_target > 0

total_nats = loss_nats[counted].sum()
total_bytes = bytes_per_target[counted].sum()
bpb = total_nats / (math.log(2) * total_bytes)

print("target ids:       ", target_ids.tolist())
print("loss nats:        ", loss_nats.tolist())
print("bytes per target: ", bytes_per_target.tolist())
print("counted targets:  ", counted.tolist())
print()
print(f"total nats:  {total_nats.item():.3f}")
print(f"total bytes: {total_bytes.item()}")
print(f"BPB:         {bpb.item():.3f}")

## Step 8. Real BPB command

This is the command you ran after the `d4` run:

```bash
python -m scripts.base_eval \
  --device-type=cuda \
  --device-batch-size=1 \
  --split-tokens=2048 \
  --eval=bpb,sample \
  --model-tag=d4 \
  --step=2000
```

`--split-tokens` controls how much train/val text to sample for BPB. Larger is more stable but slower.

The in-training validation number and a separate `base_eval.py` number can differ because they may evaluate different token windows or amounts of text.

In [ ]:
if selected_model_tag is None:
    print("No checkpoint found, so this is a template command:")
    tag = "d4"
    step = 2500
else:
    tag = selected_model_tag
    step = selected_step

device_type = "cuda" if torch.cuda.is_available() else "cpu"
print("Run this from the repo root if you want real BPB + sample eval:")
print()
print("python -m scripts.base_eval \\")
print(f"  --device-type={device_type} \\")
print("  --device-batch-size=1 \\")
print("  --split-tokens=2048 \\")
print("  --eval=bpb,sample \\")
print(f"  --model-tag={tag} \\")
print(f"  --step={step}")

## Step 9. What CORE evaluation is doing

CORE is not generation. It is closer to asking:

```text
Which answer continuation does the model find least surprising?
```

For multiple choice tasks, nanochat renders one prompt per choice, runs the model, computes the average loss on the answer continuation, and picks the lowest-loss choice.

Toy example:

```text
Prompt: The capital of France is ___
Choice A: Paris    average loss 0.7
Choice B: protein  average loss 3.2
Choice C: Friday   average loss 2.8

Prediction: Paris, because 0.7 is the lowest loss
```

In [ ]:
choices = ["Paris", "protein", "Friday"]
average_losses = torch.tensor([0.7, 3.2, 2.8])

prediction = int(torch.argmin(average_losses).item())

print("Toy CORE-style multiple-choice scoring:")
for i, (choice, loss) in enumerate(zip(choices, average_losses.tolist())):
    marker = "<-- chosen" if i == prediction else ""
    print(f"  {choice:8s} average continuation loss: {loss:.2f} {marker}")

print()
print("The model is not directly saying 'Paris'.")
print("We are using its losses to choose which continuation it finds most likely.")

## Step 10. Why `--max-per-task=1` failed for CORE

Some CORE tasks are few-shot tasks, for example `10-shot`.

That means each evaluated example needs additional examples to place in the prompt.

If you run:

```bash
--max-per-task=1
```

then a `10-shot` task has only one example available, but it needs ten examples for the few-shot context. Python correctly complains:

```text
ValueError: Sample larger than population
```

For quick smoke tests, either skip CORE:

```bash
--eval=bpb,sample
```

or use enough examples:

```bash
--eval=core --max-per-task=20
```

In [ ]:
print("Safe quick eval commands:\n")
print("1. BPB + samples, usually the best quick check:")
print("python -m scripts.base_eval --device-type=cuda --eval=bpb,sample --model-tag=d4 --step=2500 --split-tokens=2048 --device-batch-size=1")
print()
print("2. CORE smoke test, enough examples for few-shot tasks:")
print("python -m scripts.base_eval --device-type=cuda --eval=core --model-tag=d4 --step=2500 --max-per-task=20 --device-batch-size=1")
print()
print("3. Full-ish eval is slower:")
print("python -m scripts.base_eval --device-type=cuda --eval=core,bpb,sample --model-tag=d4 --step=2500")

## Step 11. How to read your small-run samples

Your progression looked like this:

```text
step 50:   repeated 'the the the...' / token soup
step 2000: English-like nonsense
step 2500: slightly smoother English-like nonsense
```

That is expected.

A tiny base model first learns:

- common words
- punctuation
- phrase shapes
- paragraph-like structure

It learns reliable facts and reasoning much later, and only if the model/data/training budget are large enough.

In [ ]:
steps = 2500
total_batch_size = 2048
tokens_seen = steps * total_batch_size

print(f"Toy run tokens seen: {tokens_seen:,}")
print("That is tiny for language modeling.")
print()
print("This is why the model can learn English-looking structure")
print("while still answering 'France', 'gold', and math prompts incorrectly.")

## Step 12. Before an expensive run: choose where artifacts live

The most important practical detail is `NANOCHAT_BASE_DIR`.

Most expensive outputs are **not** saved inside the git repo. They are saved under:

```text
$NANOCHAT_BASE_DIR
```

If this points to storage that disappears when the machine/pod stops, you can lose the expensive part of the run.

Current `runs/speedrun.sh` uses:

```bash
export NANOCHAT_BASE_DIR="${NANOCHAT_BASE_DIR:-$HOME/.cache/nanochat}"
```

So for a cloud machine, set it to a persistent disk before launching:

```bash
export NANOCHAT_BASE_DIR=/workspace/nanochat-cache
mkdir -p "$NANOCHAT_BASE_DIR"
screen -L -Logfile "$NANOCHAT_BASE_DIR/speedrun.log" -S speedrun bash runs/speedrun.sh
```

The exact persistent path depends on the provider. The idea is not the path. The idea is: checkpoints and tokenizer must land somewhere that survives shutdown.

On RunPod, the clean flow is:

1. Template bootstrap prepares the pod and clones the repo. In this repo, `runs/runpod_template_start.sh` is the pasteable RunPod command for that layer.
2. You run `bash runs/preflight_artifacts.sh` once to prove `NANOCHAT_BASE_DIR` receives files.
3. Only then you manually launch `bash runs/speedrun.sh` inside `screen`.

Cost note: a running pod can cost money even when training is not running. Do the bootstrap only when you are ready to run preflight/speedrun, and stop or terminate the pod after preserving artifacts.

In [ ]:
speedrun_path = repo_root / "runs" / "speedrun.sh"
print("The speedrun script controls artifact location near the top of the file. Relevant lines:\n")
for line in speedrun_path.read_text().splitlines()[:25]:
    if "NANOCHAT_BASE_DIR" in line or "OMP_NUM_THREADS" in line:
        print(line)

print("\nCurrent notebook base dir:")
print(get_base_dir())
print("\nBefore a paid run, make sure this location is on persistent storage.")

## Step 13. After an expensive run: preserve the right files

The expensive artifacts are the ones that are slow or impossible to recreate without paying again.

Keep these:

| Path under `NANOCHAT_BASE_DIR` | Why it matters |
|---|---|
| `tokenizer/tokenizer.pkl` | tokenizer used by the model |
| `tokenizer/token_bytes.pt` | needed for BPB evaluation |
| `base_checkpoints/` | pretrained base model checkpoints |
| `chatsft_checkpoints/` | SFT/chat checkpoints, if speedrun reached SFT |
| `chatrl_checkpoints/` | RL checkpoints, if you later run RL |
| `base_eval/` | CORE CSVs from `base_eval.py` |
| `report/` and `report.md` | generated training report and run summary |
| `identity_conversations.jsonl` | small SFT identity data used by speedrun |

Usually okay to redownload or regenerate:

- `base_data_climbmix/`
- `eval_bundle/`
- `.venv/`
- downloaded package caches

Also preserve the code state. Record the branch and commit hash you trained from. Otherwise the checkpoint exists but the exact code that produced it becomes fuzzy.

In [ ]:
import shlex

def human_bytes(n):
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}"
        value /= 1024

def tree_size(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    if path.is_dir():
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
    return 0

base_dir = Path(get_base_dir())
important = [
    "tokenizer",
    "base_checkpoints",
    "chatsft_checkpoints",
    "chatrl_checkpoints",
    "base_eval",
    "report",
    "identity_conversations.jsonl",
]

existing = []
print(f"Inspecting: {base_dir}\n")
for name in important:
    path = base_dir / name
    if path.exists():
        existing.append(name)
        print(f"KEEP    {name:28s} {human_bytes(tree_size(path)):>10s}")
    else:
        print(f"missing {name}")

print("\nArchive command for the artifacts that exist right now:")
if existing:
    archive_name = "nanochat-artifacts-$(date +%Y%m%d-%H%M%S).tar.gz"
    quoted = " ".join(shlex.quote(x) for x in existing)
    print(f"cd {shlex.quote(str(base_dir))}")
    print(f"tar -czf {archive_name} {quoted}")
else:
    print("No expensive artifacts found yet in this base dir.")

print("\nAlso save code state:")
print("git status --short")
print("git rev-parse --abbrev-ref HEAD")
print("git rev-parse HEAD")
print("git diff > nanochat-local-changes.patch  # if you have uncommitted changes")

## Step 14. What this means before a bigger GPU run

Before a larger training run, you now know how to check:

- Did checkpoint loading work?
- Did samples improve qualitatively?
- Did validation BPB go down?
- Did resume continue from the saved step?
- Did eval write output where expected?

Useful places to inspect:

```text
~/.cache/nanochat/base_checkpoints/<model-tag>/
~/.cache/nanochat/base_eval/
./wandb/offline-run-.../
$NANOCHAT_BASE_DIR/report/report.md
```

Suggested learning loop:

```text
train a run -> evaluate checkpoint -> inspect samples/BPB -> resume or scale up
```

After this notebook, the `$100` speedrun is no longer a mysterious black box. It is the same pipeline, just bigger and distributed across more GPUs.

For the actual RunPod run, the safe sequence is:

```bash
cd /workspace/nanochat
bash runs/preflight_artifacts.sh
screen -L -Logfile "$NANOCHAT_BASE_DIR/speedrun.log" -S speedrun bash runs/speedrun.sh
```

The preflight is cheap. The speedrun is the expensive part.